In [ ]:
# LSTM classification on IMDB Dataset of 50K Movie Reviews

import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping

# 1. Load dataset
df = pd.read_csv("IMDB Dataset.csv")

# 2. Show basic information
print(df.head())
print(df["sentiment"].value_counts())

# 3. Clean text
def clean_text(text):
    text = re.sub(r"<.*?>", "", text)
    text = text.lower()
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["review"] = df["review"].apply(clean_text)

# 4. Convert labels to numbers
df["sentiment"] = df["sentiment"].map({"negative": 0, "positive": 1})

# 5. Split text and labels
X_text = df["review"].values
y = df["sentiment"].values

# 6. Train, test split
X_train_text, X_test_text, y_train, y_test = train_test_split(X_text, y, test_size=0.20, random_state=42, stratify=y)

# 7. Tokenization
vocab_size = 10000
max_len = 200
embedding_dim = 100

# The tokenizer keeps the most frequent 10,000 words from the training set.
tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>") #<OOV>: out-of-vocabulary token
tokenizer.fit_on_texts(X_train_text)

X_train_seq = tokenizer.texts_to_sequences(X_train_text)
X_test_seq = tokenizer.texts_to_sequences(X_test_text)

# 8. Padding and truncation
X_train = pad_sequences(X_train_seq, maxlen=max_len, padding="post", truncating="post")
X_test = pad_sequences(X_test_seq, maxlen=max_len, padding="post", truncating="post")

# 9. Show shapes
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

# 10. Build LSTM model
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len, mask_zero=True),
*********************
    Dropout(0.5),
    Dense(1, activation="sigmoid")
])

# 11. Compile model
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

model.summary()

# 12. Early stopping
early_stop = EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)

# 13. Train model
history = model.fit(X_train, y_train, validation_split=0.20, epochs=10, batch_size=32, callbacks=[early_stop], verbose=1)

# 14. Evaluate model
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

# 15. Predict
y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob >= 0.5).astype(int).flatten()

# 16. Extra evaluation
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=["Negative", "Positive"]))

# 17. Try one custom review
new_review = ["The movie was not good. It was boring and too long."]

new_review_clean = [clean_text(text) for text in new_review]
new_review_seq = tokenizer.texts_to_sequences(new_review_clean)
new_review_pad = pad_sequences(new_review_seq, maxlen=max_len, padding="post", truncating="post")

prediction = model.predict(new_review_pad)[0][0]

print("Prediction probability:", prediction)

if prediction >= 0.5:
    print("Predicted sentiment: Positive")
else:
    print("Predicted sentiment: Negative")

In [ ]:
new_review_pad
#The Embedding layer turns these IDs into word embeddings.

#Using Glove

In [ ]:
!pip install gensim

# LSTM classification on IMDB Dataset of 50K Movie Reviews using GloVe with gensim

import pandas as pd
import numpy as np
import re
import gensim.downloader as api
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping

# 1. Load dataset
df = pd.read_csv("IMDB Dataset.csv")

# 2. Show basic information
print(df.head())
print(df["sentiment"].value_counts())

# 3. Clean text
def clean_text(text):
    text = re.sub(r"<.*?>", "", text)
    text = text.lower()
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["review"] = df["review"].apply(clean_text)

# 4. Convert labels to numbers
df["sentiment"] = df["sentiment"].map({"negative": 0, "positive": 1})

# 5. Split text and labels
X_text = df["review"].values
y = df["sentiment"].values

# 6. Train, test split
X_train_text, X_test_text, y_train, y_test = train_test_split(X_text, y, test_size=0.20, random_state=42, stratify=y)

# 7. Tokenization
vocab_size = 10000
max_len = 200
embedding_dim = 100

# The tokenizer keeps the most frequent 10,000 words from the training set.
tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>") # <OOV>: out-of-vocabulary token
tokenizer.fit_on_texts(X_train_text)

*********************
X_test_seq = tokenizer.texts_to_sequences(X_test_text)

# 8. Padding and truncation
X_train = pad_sequences(X_train_seq, maxlen=max_len, padding="post", truncating="post")
*********************

# 9. Show shapes
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

# 10. Load GloVe embeddings using gensim
glove_model = api.load("glove-wiki-gigaword-100")

print("GloVe model loaded")

# 11. Create embedding matrix
word_index = tokenizer.word_index

embedding_matrix = np.zeros((vocab_size, embedding_dim))

for word, i in word_index.items():
    if i < vocab_size: #after fit_on_texts, words are usually indexed by frequency, so the most frequent words receive smaller IDs.
        if word in glove_model:
            embedding_matrix[i] = glove_model[word]

print("Embedding matrix shape:", embedding_matrix.shape)

# 12. Build LSTM model
model = Sequential([
    Input(shape=(max_len,)),
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, weights=[embedding_matrix], mask_zero=True, trainable=False),
    LSTM(64, activation="tanh"),
    Dropout(0.5),
    Dense(1, activation="sigmoid")
])

# 13. Compile model
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

model.summary()

# 14. Early stopping
early_stop = EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)

# 15. Train model
history = model.fit(X_train, y_train, validation_split=0.20, epochs=10, batch_size=32, callbacks=[early_stop], verbose=1)

# 16. Evaluate model
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

# 17. Predict
y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob >= 0.5).astype(int).flatten()

# 18. Extra evaluation
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=["Negative", "Positive"]))

# 19. Try one custom review
new_review = ["The movie was not good. It was boring and too long."]

new_review_clean = [clean_text(text) for text in new_review]
new_review_seq = tokenizer.texts_to_sequences(new_review_clean)
new_review_pad = pad_sequences(new_review_seq, maxlen=max_len, padding="post", truncating="post")

prediction = model.predict(new_review_pad)[0][0]

print("Prediction probability:", prediction)

if prediction >= 0.5:
    print("Predicted sentiment: Positive")
else:
    print("Predicted sentiment: Negative")

#RNN for Text Generation

In [ ]:
# LSTM text generation using Tiny Shakespeare, word-level/token-level version

import tensorflow as tf
import numpy as np
import re

# 1. Download Tiny Shakespeare dataset
path_to_file = tf.keras.utils.get_file("shakespeare.txt", "https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt")

# 2. Read text
text = open(path_to_file, "rb").read().decode(encoding="utf-8")

print("Length of text:", len(text))
print(text[:500])

# 3. Clean text
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

text_clean = clean_text(text)

print("Cleaned text sample:")
print(text_clean[:500])

# 4. Split text into word tokens
tokens = text_clean.split()

print("Number of tokens:", len(tokens))
print("First 50 tokens:")
print(tokens[:50])

# 5. Create vocabulary
vocab = sorted(set(tokens))

print("Vocabulary size:", len(vocab))
print("First 50 vocabulary words:")
print(vocab[:50])

# 6. Create word-to-ID and ID-to-word mappings
*********************
index_to_word = np.array(vocab)

# 7. Convert all tokens into integer IDs
text_as_int = np.array([word_to_index[word] for word in tokens])

print("First 50 token IDs:")
print(text_as_int[:50])

# 8. Choose sequence length
seq_length = 10

# 9. Create input and target sequences
word_dataset = tf.data.Dataset.from_tensor_slices(text_as_int)

sequences = word_dataset.batch(seq_length + 1, drop_remainder=True)

# Show the first 3 sequences
for seq in sequences.take(3):
    print(seq.numpy())

def split_input_target(chunk):
*********************
*********************
    return input_text, target_text

dataset = sequences.map(split_input_target)

# 10. Show one training example
for input_example, target_example in dataset.take(1):
    print("Input sequence:")
    print(" ".join(index_to_word[input_example.numpy()]))

    print("\nTarget sequence:")
    print(" ".join(index_to_word[target_example.numpy()]))

# 11. Batch and shuffle data
batch_size = 64
buffer_size = 10000

dataset = dataset.shuffle(buffer_size).batch(batch_size, drop_remainder=True).prefetch(tf.data.AUTOTUNE)
# batch(batch_size, drop_remainder=True): It simply forms batches in order, and if the last batch is incomplete, it removes it.
# shuffle(buffer_size): Keep 10,000 (buffer_size) examples in a buffer, randomly pick from them, then refill the buffer.
# prefetch(tf.data.AUTOTUNE) makes training faster by preparing the next batch while the model is training on the current batch.

# 12. Model parameters
vocab_size = len(vocab)
embedding_dim = 128
lstm_units = 256

# 13. Build LSTM model
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(input_dim=vocab_size, output_dim=embedding_dim),
    tf.keras.layers.LSTM(lstm_units, return_sequences=True),
    tf.keras.layers.Dense(vocab_size)
])

# 14. Compile model
model.compile(optimizer="adam", loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True))

model.summary()

# 15. Train model
epochs = 10

history = model.fit(dataset, epochs=epochs)

# 16. Generate text
def generate_text(model, start_string, num_generate=50, temperature=1.0):
    # Clean the starting text the same way we cleaned the training text
    start_string = clean_text(start_string)

    # Split the starting text into word tokens
    input_tokens = start_string.split()

    # This list will store the integer IDs of the starting words
    input_eval = []

    # Convert each starting word into its integer ID using the training vocabulary
    for word in input_tokens:
        if word in word_to_index:
            input_eval.append(word_to_index[word])

    # If none of the starting words exist in the vocabulary, start with a common word
    if len(input_eval) == 0:
        input_eval = [word_to_index["the"]]

    # Add a batch dimension because the model expects input shape: (batch_size, sequence_length)
    # Example: [12, 45, 8] becomes [[12, 45, 8]]
    input_eval = tf.expand_dims(input_eval, 0)

    # Store the original starting words so we can add generated words after them
    generated_words = input_tokens.copy()

    # Generate one word at a time
    for i in range(num_generate):
        # Predict the next-word scores for every position in the current input sequence
        predictions = model(input_eval)

        # Keep only the prediction from the last time step
        # Shape changes from (1, sequence_length, vocab_size) to (1, vocab_size)
        # This line takes the model’s prediction for only the last word position.
*********************

        # Control randomness:
        # lower temperature gives safer, more predictable words
        # higher temperature gives more creative, more random words
*********************

        # Randomly sample one word ID from the predicted probability distribution
        predicted_id = tf.random.categorical(predictions, num_samples=1)[-1, 0].numpy()

        # Convert the predicted word ID back into an actual word
        predicted_word = index_to_word[predicted_id]

        # Add the predicted word to the generated text
        generated_words.append(predicted_word)

        # Add the predicted word ID to the input sequence
        # This lets the model use its own generated word to predict the next word
        input_eval = tf.concat([input_eval, tf.expand_dims([predicted_id], 0)], axis=1)

        # Keep only the last seq_length tokens, so the input size stays controlled
        input_eval = input_eval[:, -seq_length:]

    # Return the starting words plus all generated words as one text string
    return " ".join(generated_words)

# 17. Try text generation
generated_text = generate_text(model, start_string="romeo", num_generate=100, temperature=0.8)

print(generated_text)

In [ ]:
print(word_to_index)

In [ ]:
index_to_word

In [ ]:
text_as_int

In [ ]:
# Show the first 3 sequences
for seq in sequences.take(3):
    print(seq.numpy())